### 实验前的环境准备

这一格不是正式建模代码，而是实验开始前的“环境体检”。

为什么要先运行它？

1. 同一份 notebook 换一台电脑、换一个 Python 环境后，常见问题不是算法写错，而是缺少库。
2. 如果一开始不先检查，后面往往会在 `import`、画图、读文件或训练模型时突然报错，初学者很难判断问题到底出在代码还是环境。
3. 现在这一格已经升级为“先检查、再自动安装”，目的就是把环境问题尽量提前解决。

运行后你会看到几类信息：

1. 当前 notebook 实际使用的是哪个 Python 解释器。
2. 已经检测到哪些核心库。
3. 如果有缺失库，系统会尝试自动安装。
4. 如果安装完成后仍未生效，通常只需要重启内核，再从第 1 格重新运行。

可以把这一格理解成：正式做实验前，先把工具箱点一遍，缺什么先补什么。这样后面的每一步更容易顺利完成，也更符合真实开发中的工作流程。

In [ ]:
import importlib
import subprocess
import sys

required_packages = {
    'pandas': 'pandas',
    'numpy': 'numpy',
    'torch': 'torch',
    'sklearn': 'scikit-learn',
    'matplotlib': 'matplotlib',
    'plotly': 'plotly',
    'nbformat': 'nbformat',
    'tqdm': 'tqdm',
    'openpyxl': 'openpyxl',
}

note_lines = [
    '说明：本教程需要读取 xlsx 文件，因此需要 openpyxl。',
    '说明：本教程使用 Plotly 在 notebook 中显示三维图，因此建议同时具备 nbformat。',
]


def is_module_available(module_name):
    return importlib.util.find_spec(module_name) is not None


installed_modules = []
missing_packages = []
for module_name, package_name in required_packages.items():
    if is_module_available(module_name):
        installed_modules.append(module_name)
    else:
        missing_packages.append(package_name)

print('当前 Python 解释器:', sys.executable)
print('已检测到的模块:', ', '.join(installed_modules) if installed_modules else '无')

if missing_packages:
    print('\n检测到缺失依赖，开始自动安装:')
    print('pip install ' + ' '.join(missing_packages))
    try:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing_packages])
        importlib.invalidate_caches()
        still_missing = [
            package_name
            for module_name, package_name in required_packages.items()
            if not is_module_available(module_name)
        ]
        if still_missing:
            print('\n以下依赖安装后仍未检测到，请重启内核后重试:')
            print(', '.join(still_missing))
        else:
            print('\n依赖已自动安装完成，可以继续运行本教程。')
    except subprocess.CalledProcessError as error:
        print(f'\n自动安装失败，返回码: {error.returncode}')
        print('请手动执行:')
        print('pip install ' + ' '.join(missing_packages))
else:
    print('\n依赖检查通过，可以继续运行本教程。')

for note in note_lines:
    print(note)

print('如果你已经安装过，但这里仍显示缺失，通常是因为当前 notebook 内核和安装库使用的 Python 环境不是同一个。')
print('如果刚完成自动安装，后续单元格仍报导入错误，重启内核后再从头运行一次。')

### 医学公开数据集回归实践

本实验将原来的房价预测案例替换为公开、经典、适合初学者的医学回归任务，同时把操作步骤整理为与常规房价回归教程一致的节奏：先读取本地文件，再检查原始表、完成数据清洗与特征工程，最后再进行 PyTorch 建模与结果分析。

- 数据集：Medical Insurance Cost Personal Dataset（公开医学费用数据集）
- 本地文件：medical_insurance_cost.xlsx
- 任务：根据年龄、BMI、子女数量、吸烟情况、地区等信息预测医疗费用
- 数据集简介：该数据集记录了个人基本信息与医疗保险费用之间的关系，常用于回归教学与特征影响分析
- 优点：公开可得、样本量与原实验接近、字段难度适中、与常规房价回归共享相同建模框架，适合横向比较不同任务

#### 本实验建议关注 4 个问题

1. 原始数据中有没有缺失值、重复值、类别字段？
2. 为什么要构造 family_size、age_group、bmi_level 这类衍生特征？
3. 为什么医疗费用要先做对数变换，再送入神经网络？
4. 如果修改学习率、训练轮数或隐藏层宽度，结果会发生什么变化？

### 开始前先建立回归概念框架

为了让医疗版和常规版的回归教程能直接对照，先把这两个实验共享的几个核心概念说清楚：

1. 回归任务预测的是连续数值，本章里分别是医疗费用和房屋总价，而不是“属于哪一类”。
2. MLP 可以理解成一组不断做加权计算的神经元，隐藏层负责从多个字段里提炼组合规律。
3. 回归模型常用 `MSELoss()` 来衡量“预测值离真实值有多远”，误差越大，模型就越需要继续调整参数。
4. `MAE` 反映平均误差大小，`R²` 反映模型解释整体变化趋势的能力，这两个指标要结合着看。
5. `learning_rate` 决定每次参数更新迈多大步，`epochs` 决定模型反复练习多少轮；训练过少可能欠拟合，训练过头也可能出现过拟合。

先建立这套概念框架，后面再看代码，就更容易把“参数怎么调”和“模型为什么这样变”对应起来。

### 第一步：准备实验工具

这一格可以理解成正式做实验前的“工具箱准备”。

这里导入的库大致分成几类：

1. `pandas`、`numpy`：负责读表、整理数据、做基础计算。
2. `torch`：负责搭建和训练神经网络。
3. `sklearn`：负责数据划分、标准化、PCA 降维和评价指标。
4. `matplotlib`、`plotly`：负责把结果画出来，帮助我们观察模型表现。
5. `tqdm`：负责显示训练进度条。

零基础时不需要一上来就背所有库名，更重要的是先知道：后面每一步操作，其实都在调用这里准备好的工具。

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, median_absolute_error, r2_score
import matplotlib.pyplot as plt
from matplotlib import font_manager
import plotly.express as px
import plotly.io as pio
from tqdm import trange

def configure_matplotlib_for_cjk():
    preferred_fonts = [
        "PingFang SC",
        "Hiragino Sans GB",
        "Heiti SC",
        "STHeiti",
        "Songti SC",
        "Arial Unicode MS",
        "Microsoft YaHei",
        "SimHei",
        "Noto Sans CJK SC",
        "Source Han Sans SC",
        "WenQuanYi Zen Hei",
        "DejaVu Sans",
    ]
    available_fonts = {font.name for font in font_manager.fontManager.ttflist}
    for font_name in preferred_fonts:
        if font_name in available_fonts:
            plt.rcParams["font.family"] = font_name
            break
    plt.rcParams["axes.unicode_minus"] = False

configure_matplotlib_for_cjk()
pio.renderers.default = "plotly_mimetype"

### 第二步：认识原始数据

初学者最容易犯的错误，就是文件一读进来就直接开始训练模型。

这一步真正的目标不是“赶快跑代码”，而是先回答几个最基本的问题：

1. 一共有多少样本、多少字段？
2. 哪些字段是数字，哪些字段是文本？
3. 有没有缺失值、重复值？
4. 目标变量到底是什么？

可以把这一步理解成“看病前先量体温、问病史”，如果连数据长什么样都没弄清楚，后面的模型结果就很难解释。

In [ ]:
# 从本地 Excel 文件读取公开医学回归数据集，并先检查原始表结构
df = pd.read_excel("medical_insurance_cost.xlsx")

print("数据集名称: Medical Insurance Cost Personal Dataset")
print("任务类型: 医学回归")
print("原始样本数, 原始字段数:", df.shape)
print("\n原始字段信息:")
print(df.info())
print("\n原始缺失值统计:")
print(df.isnull().sum())
print("\n原始重复行数量:", df.duplicated().sum())
print("\n原始数据前 5 行:")
print(df.head())

print("\n教学提示：")
print("1. 先看哪些字段是数值型，哪些字段是类别型。")
print("2. 如果直接把 sex、smoker、region 送入模型，神经网络并不能直接理解文本，需要后续编码。")
print("3. 即使这个公开数据集很干净，也要保留“先检查再建模”的好习惯。")

### 原始特征与补充特征含义

在真正开始特征工程前，先把每个字段在现实里代表什么说清楚。这样后面看到 `family_size`、`age_group` 这类新特征时，学生不会觉得它们是“凭空多出来的一列”。

**原始特征含义：**

1. `age`：年龄，反映患者所处的人生阶段。
2. `sex`：性别，表示样本的基本人口学属性。
3. `bmi`：身体质量指数，常用于衡量体重是否偏低、正常或超重。
4. `children`：需要共同承担家庭医疗支出的子女人数。
5. `smoker`：是否吸烟，是影响健康风险和医疗费用的重要行为变量。
6. `region`：所在地区，不同地区的医疗资源和费用水平可能不同。
7. `charges`：个人医疗费用，也是本任务要预测的目标值。

**补充特征含义：**

1. `family_size`：家庭规模，这里用“子女人数 + 1”近似，帮助模型从家庭负担角度理解费用差异。
2. `age_group`：年龄分组，把连续年龄划成更容易观察的阶段区间。
3. `bmi_level`：BMI 分层，把“偏低、正常、超重、肥胖”这类健康状态显式表示出来。
4. `smoker_flag`：把“是否吸烟”转成 0/1 数字，便于模型直接计算。

可以引导学生思考：这些补充特征并没有引入外部新数据，而是把原始字段换成了更利于模型学习的表达方式。

### 第三步：构造衍生特征并完成编码
**为什么要进行这一步？**
1. **清洗（Cleaning）**：
   - 原始表格即使看起来很干净，也可能包含重复行或不适合直接建模的记录
   - 先清洗，才能保证模型学习的是更稳定的数据规律
   - 这一步的目标不是把数据“修饰漂亮”，而是减少无意义噪声

2. **构造衍生特征（Feature Engineering）**：
   - 原始字段不一定能直接表达我们真正关心的健康状态
   - 通过增加 `family_size`、`age_group`、`bmi_level` 等特征，可以把连续数值整理成更容易观察和学习的结构
   - 这些新特征并不是凭空创造信息，而是把原始字段重新组织成更适合模型理解的表达方式

3. **编码（Encoding）**：
   - 神经网络只能处理数值型输入，不能直接理解 `sex`、`smoker`、`region` 这样的文本标签
   - 因此需要把类别变量转换成数值形式，例如独热编码
   - 编码后的每一列都可以看作“某种属性是否成立”的数字提示

**总结**：这一格的核心，是先把现实世界中的人口学和健康信息翻译成模型可以计算的数字特征。

In [ ]:
# 数据清洗、特征工程与类别编码
df1 = df.drop_duplicates().dropna().copy()

# 增加少量医学场景下容易理解的衍生特征，让流程更接近原实验的特征整理过程
df1["family_size"] = df1["children"] + 1
df1["age_group"] = pd.cut(
    df1["age"], bins=[17, 25, 35, 50, 65], labels=["18-25", "26-35", "36-50", "51-64"]
)
df1["bmi_level"] = pd.cut(
    df1["bmi"], bins=[0, 18.5, 24, 28, 100], labels=["偏低", "正常", "超重", "肥胖"]
)
df1["smoker_flag"] = df1["smoker"].map({"no": 0, "yes": 1})

print("清洗后样本数, 字段数:", df1.shape)
print("\n清洗后缺失值统计:")
print(df1.isnull().sum())
print("\n关键数值字段描述统计:")
print(df1[["age", "bmi", "children", "family_size", "charges"]].describe().round(2))

df_model = pd.get_dummies(
    df1, columns=["sex", "smoker", "region", "age_group", "bmi_level"], drop_first=True
)
feature_columns = [column for column in df_model.columns if column != "charges"]
X = df_model[feature_columns].astype(float).values
y_raw = df_model[["charges"]].values

print("\n编码后建模字段数:", len(feature_columns))
print("部分建模字段:", feature_columns[:12])

### 第四步：完成标准化、数据划分与张量化
**为什么要进行这一步？**
1. **标准化（Standardization）**：
   - 不同特征的量纲差异很大，例如年龄、BMI、医疗费用的数值范围并不在同一尺度上
   - 标准化可以把这些特征拉回到更接近的数值区间，避免某些大数值字段过度主导训练
   - 对回归任务来说，这一步通常会让训练过程更稳定

2. **训练集 / 测试集划分（Train/Test Split）**：
   - 训练集负责让模型学习规律，测试集负责检查模型是不是学会了可泛化的模式
   - 如果不分开，模型可能只是把答案记住了，而不是学到了规律

3. **张量化（Tensorization）**：
   - PyTorch 使用张量作为基本数据结构
   - 把 NumPy 数组转换成张量后，模型才能直接参与前向计算、反向传播和参数更新

**总结**：前一步是在整理“学什么”，这一步是在整理“怎么学”，让数据真正进入神经网络训练流程。

In [ ]:
# 划分训练集和测试集，并完成标准化与张量化
y_log = np.log1p(y_raw)

scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y_log)

X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(
    X_scaled, y_scaled, test_size=0.2, random_state=42
)

X_train = torch.tensor(X_train_np, dtype=torch.float32)
y_train = torch.tensor(y_train_np, dtype=torch.float32)
X_test = torch.tensor(X_test_np, dtype=torch.float32)
y_test = torch.tensor(y_test_np, dtype=torch.float32)

print("训练集形状:", X_train.shape, y_train.shape)
print("测试集形状:", X_test.shape, y_test.shape)

#### 特征降维可视化

 > 这一步不是为了替代建模，而是为了帮助你先观察：现有特征是否已经让不同费用层次的样本出现一定分布差异。

 > 现在改为三维交互式散点图，你可以在 notebook 输出中直接旋转、缩放和悬停观察不同费用区间样本的空间分布。

 > 如果三维投影中不同费用区间仍然大面积重叠，就说明后续还可以考虑继续设计更有区分度的特征。

In [ ]:
# 用 PCA 把多维特征投影到三维空间，观察不同费用区间的样本分布
charge_bins = pd.qcut(df1['charges'], q=4, duplicates='drop')
pca = PCA(n_components=3)
X_pca = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame({
    '主成分1': X_pca[:, 0],
    '主成分2': X_pca[:, 1],
    '主成分3': X_pca[:, 2],
    '真实费用区间': charge_bins.astype(str),
    '真实医疗费用': df1['charges'].round(2),
    '吸烟标记': df1['smoker_flag'],
    'BMI': df1['bmi'].round(2),
    '年龄': df1['age'],
})

fig = px.scatter_3d(
    pca_df,
    x='主成分1',
    y='主成分2',
    z='主成分3',
    color='真实费用区间',
    hover_data=['真实医疗费用', '年龄', 'BMI', '吸烟标记'],
    title='医学回归任务：特征降维后的三维交互分布',
    opacity=0.7,
    color_discrete_sequence=px.colors.qualitative.Set2,
 )
fig.update_traces(marker=dict(size=4))
fig.update_layout(margin=dict(l=0, r=0, t=50, b=0))
fig.show()

explained_ratio = pca.explained_variance_ratio_
print('前三个主成分的方差解释率:', np.round(explained_ratio, 4))
print('累计解释率:', round(float(explained_ratio.sum()), 4))
print('教学提示：你可以旋转图形，观察高费用样本是否在三维空间里形成相对集中的区域。')

In [ ]:
# 学生可在这里修改训练参数，再重新运行后续单元
student_config = {
    "hidden_dims": [32, 16],
    "learning_rate": 0.001,
    "epochs": 1000,
}

print("当前实验参数:", student_config)
print("你可以尝试修改 hidden_dims、learning_rate、epochs，然后重新运行后面的训练单元。")
print("说明：根据小范围参数搜索，[32, 16] 在当前医学回归数据上比 [64, 32] 更稳，适合作为新的默认起点。")

#### 可调参数实验

下面这个单元故意把部分训练参数单独拿出来，方便同学们修改后重新运行。

当前默认结构已经根据小范围参数搜索调整为更稳的 `[32, 16]`，建议初学者先只改 1 个参数：
- 把 epochs 从 1000 改到 600 或 1400
- 把 learning_rate 从 0.001 改到 0.0005 或 0.002
- 把 hidden_dims 从 [32, 16] 改回 [64, 32]，观察更宽网络是否一定更好

每次只改一个参数，更容易观察模型变化的原因。

### 第五步：开始训练 MLP 回归模型

这里正式进入“让模型学习”的阶段。

这一格可以拆成两部分来看：

1. 先定义网络结构：输入层接收特征，隐藏层负责提炼模式，输出层给出预测值。
2. 再执行训练循环：模型先预测，再计算误差，再根据误差反向调整参数。

如果把训练过程想得更形象一些，可以理解成：
模型先答题，老师告诉它错了多少，它再一点点改正，重复很多轮之后，答案才会越来越接近真实值。

In [ ]:
# 定义模型，训练并验证模型
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dims):
        super().__init__()
        layers = []
        previous_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(previous_dim, hidden_dim))
            layers.append(nn.ReLU())
            previous_dim = hidden_dim
        layers.append(nn.Linear(previous_dim, 1))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        return self.model(x)

hidden_dims = student_config.get("hidden_dims", [64, 32])
learning_rate = student_config.get("learning_rate", 0.001)
epochs = student_config.get("epochs", 1000)

mlp = MLP(X_train.shape[1], hidden_dims)
criterion = nn.MSELoss()
optimizer = optim.Adam(mlp.parameters(), lr=learning_rate)

for epoch in trange(epochs, desc="Training Epochs"):
    mlp.train()
    optimizer.zero_grad()
    output = mlp(X_train)
    loss = criterion(output, y_train)
    loss.backward()
    optimizer.step()

mlp.eval()
with torch.no_grad():
    pred = mlp(X_test)
    y_pred_log = scaler_y.inverse_transform(pred.numpy())
    y_true_log = scaler_y.inverse_transform(y_test.numpy())
    y_pred = np.expm1(y_pred_log).reshape(-1)
    y_true = np.expm1(y_true_log).reshape(-1)

    mae = mean_absolute_error(y_true, y_pred)
    mdae = median_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"平均绝对误差 MAE: {mae:.2f}")
    print(f"中位绝对误差 MdAE: {mdae:.2f}")
    print(f"R²: {r2:.3f}")
    print("\n参数解释：")
    print(f"hidden_dims = {hidden_dims} 表示隐藏层结构")
    print(f"learning_rate = {learning_rate} 表示每一步更新幅度")
    print(f"epochs = {epochs} 表示完整遍历训练集的次数")
    print("默认设置与原始房价回归教程保持一致，便于比较不同数据任务下模型表现。")

    plt.figure(figsize=(8, 6))
    plt.scatter(y_true, y_pred, alpha=0.7, color="#2f7d6d")
    min_value = min(y_true.min(), y_pred.min())
    max_value = max(y_true.max(), y_pred.max())
    plt.plot([min_value, max_value], [min_value, max_value], "r--", label="理想预测线")
    plt.xlabel("真实医疗费用")
    plt.ylabel("模型预测费用")
    plt.title("Insurance 公开数据集：真实值与预测值对比")
    plt.legend()
    plt.tight_layout()
    plt.show()

    error = np.abs(y_true - y_pred)
    worst_index = int(np.argmax(error))
    print("\n误差较大的一个样本：")
    print("真实值:", float(y_true[worst_index]))
    print("预测值:", float(y_pred[worst_index]))
    print("绝对误差:", float(error[worst_index]))
    print("说明：医疗费用存在高额样本，少数极端值会明显拉大最大绝对误差。")
    print("提示：该模型只用于教学演示，不能直接用于真实医疗收费判断。")

### 第六步：分析测试集误差分布

回归任务里，一个平均误差只能告诉我们“整体大概差多少”，但它不能回答更具体的问题：

1. 模型是在高费用样本上误差更大，还是低费用样本上更稳定？
2. 误差是均匀分布的，还是集中在少数区间？
3. 模型有没有系统性地高估或低估某些人群的医疗费用？

这一步的意义，就是把“一个总成绩”拆开看成“不同题型的得分情况”，帮助我们更细致地理解模型能力。

In [ ]:
# 误差分布可视化：按测试集真实费用区间统计误差范围
abs_error = np.abs(y_true - y_pred)
correlation = float(np.corrcoef(y_true, y_pred)[0, 1])

error_df = pd.DataFrame({
    '真实费用': y_true,
    '预测费用': y_pred,
    '绝对误差': abs_error,
})
error_df['费用区间'] = pd.qcut(
    error_df['真实费用'],
    q=4,
    duplicates='drop',
    precision=1,
 )

error_summary = error_df.groupby('费用区间', observed=False)['绝对误差'].agg(
    ['count', 'mean', 'median', lambda values: np.percentile(values, 90), 'max']
)
error_summary.columns = ['样本数', '平均误差', '中位误差', '90分位误差', '最大误差']
error_summary = error_summary.round(2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_true, y_pred, alpha=0.65, color='#2f7d6d', label='测试样本')
line_min = min(y_true.min(), y_pred.min())
line_max = max(y_true.max(), y_pred.max())
axes[0].plot([line_min, line_max], [line_min, line_max], 'r--', label='理想预测线')
axes[0].set_xlabel('真实医疗费用')
axes[0].set_ylabel('预测医疗费用')
axes[0].set_title('真实费用与预测费用的相关性')
axes[0].legend()

boxplot_data = [
    error_df.loc[error_df['费用区间'] == interval_label, '绝对误差'].values
    for interval_label in error_summary.index
]
axes[1].boxplot(boxplot_data, patch_artist=True)
axes[1].set_xticks(range(1, len(error_summary.index) + 1))
axes[1].set_xticklabels([str(label) for label in error_summary.index])
axes[1].set_xlabel('测试集真实费用区间')
axes[1].set_ylabel('绝对误差')
axes[1].set_title('不同费用区间的测试误差分布')
axes[1].tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

print(f'真实费用与预测费用的相关系数: {correlation:.3f}')
print('\n不同费用区间的测试误差统计：')
print(error_summary)
print('\n教学提示：如果高费用区间的平均误差、90 分位误差和最大误差更大，说明模型在高费用样本上的稳定性更弱。')

#### 特征重要性反思与验证

> 这里使用一个适合初学者理解的置乱测试：每次只打乱一个特征，再观察测试误差是否明显上升。

> 如果打乱某个特征后 MAE 增加很多，说明当前模型比较依赖这个特征。

In [ ]:
# 用置乱测试做一个适合初学者理解的特征重要性验证
baseline_mae = float(mae)
X_test_for_importance = X_test.numpy().copy()
importance_rows = []

for feature_index, feature_name in enumerate(feature_columns):
    shuffled_X_test = X_test_for_importance.copy()
    np.random.seed(42)
    np.random.shuffle(shuffled_X_test[:, feature_index])
    shuffled_tensor = torch.tensor(shuffled_X_test, dtype=torch.float32)

    with torch.no_grad():
        shuffled_pred_log = scaler_y.inverse_transform(mlp(shuffled_tensor).numpy())
        shuffled_pred = np.expm1(shuffled_pred_log).reshape(-1)

    shuffled_mae = mean_absolute_error(y_true, shuffled_pred)
    importance_rows.append({
        '特征': feature_name,
        '打乱后MAE': round(float(shuffled_mae), 2),
        'MAE增加值': round(float(shuffled_mae - baseline_mae), 2),
    })

importance_df = pd.DataFrame(importance_rows).sort_values('MAE增加值', ascending=False)
top_importance = importance_df.head(10)
print('基线 MAE:', round(baseline_mae, 2))
print('\n特征重要性验证结果（前 10 项）：')
print(top_importance)

plt.figure(figsize=(8, 5))
plt.barh(top_importance['特征'], top_importance['MAE增加值'], color='#4c956c')
plt.gca().invert_yaxis()
plt.xlabel('打乱该特征后增加的 MAE')
plt.ylabel('特征')
plt.title('医学回归任务：置乱测试下的特征重要性')
plt.tight_layout()
plt.show()

print('教学提示：MAE 增加值越大，说明模型越依赖该特征。')
print('注意：这只是教学版的重要性验证，不等于严格的因果解释。')

In [ ]:
# 选做：快速比较几组参数，观察指标变化
trial_configs = [
    {"hidden_dims": [64, 32], "learning_rate": 0.001, "epochs": 1000},
    {"hidden_dims": [64, 32], "learning_rate": 0.001, "epochs": 1400},
    {"hidden_dims": [32, 16], "learning_rate": 0.001, "epochs": 1000},
]

print("下面的单元适合课后尝试，运行时间会比前面更长一些。")
print("你可以自行增删 trial_configs 中的参数组合。")

In [ ]:
# 选做：真正运行几组参数，比较哪组在测试集上更稳
trial_results = []

for trial_config in trial_configs:
    torch.manual_seed(42)
    trial_mlp = MLP(X_train.shape[1], trial_config["hidden_dims"])
    trial_optimizer = optim.Adam(trial_mlp.parameters(), lr=trial_config["learning_rate"])
    trial_criterion = nn.MSELoss()

    for _ in trange(trial_config["epochs"], desc=f"Trial {trial_config}", leave=False):
        trial_mlp.train()
        trial_optimizer.zero_grad()
        trial_output = trial_mlp(X_train)
        trial_loss = trial_criterion(trial_output, y_train)
        trial_loss.backward()
        trial_optimizer.step()

    trial_mlp.eval()
    with torch.no_grad():
        trial_pred = trial_mlp(X_test)
        trial_pred_log = scaler_y.inverse_transform(trial_pred.numpy())
        trial_true_log = scaler_y.inverse_transform(y_test.numpy())
        trial_pred_value = np.expm1(trial_pred_log).reshape(-1)
        trial_true_value = np.expm1(trial_true_log).reshape(-1)

    trial_results.append({
        "hidden_dims": str(trial_config["hidden_dims"]),
        "learning_rate": trial_config["learning_rate"],
        "epochs": trial_config["epochs"],
        "MAE": round(mean_absolute_error(trial_true_value, trial_pred_value), 2),
        "MdAE": round(median_absolute_error(trial_true_value, trial_pred_value), 2),
        "R2": round(r2_score(trial_true_value, trial_pred_value), 4),
    })

trial_results_df = pd.DataFrame(trial_results).sort_values(["MAE", "MdAE"])
print(trial_results_df)
print("\n说明：如果改动后的结果没有稳定优于默认配置，就不建议为了追求小幅分数波动而修改默认参数。")

#### 结果反思与动手挑战

请同学们结合本次运行结果尝试回答下面几个问题：

1. 本次运行里，模型的 $R^2$ 约为 0.82、相关系数约为 0.91，这说明模型已经学到主要趋势了吗？还存在哪些明显短板？
2. 从费用区间统计看，最高费用区间的平均误差明显高于低费用区间。为什么高费用人群更难预测？
3. 在三维交互图中，高费用样本更像形成局部簇，还是仍与普通样本部分重叠？这和高费用区间误差偏大的现象能否对应起来？
4. 从置乱测试结果看，`age`、`smoker_flag`、`smoker_yes` 为什么会排在最前面？这是否符合医学常识？
5. 为什么 `family_size`、`children` 这类衍生特征也有贡献，但明显弱于年龄和吸烟相关特征？
6. 如果把 `hidden_dims` 改小到 `[32, 16]`，或者把 `epochs` 提高到 1400，结果一定会更好吗？为什么？

<details>
<summary>提示与参考答案</summary>

提示：把三维交互图、真实值-预测值相关性图、费用区间误差统计和置乱重要性结果结合起来看，不要只盯住一个指标。

参考答案：
1. $R^2$ 约 0.82、相关系数约 0.91，说明模型已经学到主要趋势，但对高费用样本的刻画仍明显不足。
2. 高费用样本更少，而且往往受到更多表外因素影响，所以平均误差、90 分位误差和最大误差都更容易被拉大。
3. 如果高费用样本在三维图中只形成局部聚集、但仍与其他样本部分重叠，就说明现有特征只能部分解释高费用差异，这与高费用区间误差偏大是一致的。
4. `age` 和吸烟相关特征排在前面，说明模型最依赖这些高风险信息，这符合医疗费用随年龄和吸烟风险上升的常识。
5. `family_size`、`children` 这类特征提供的是补充信息，可以帮助模型细化家庭负担差异，但通常不会比年龄和吸烟更直接。
6. 更小的隐藏层可能欠拟合，更多训练轮数也可能带来过拟合，因此结果不一定单调变好。
</details>